# Scientific Python
## Central European University

## 04 Optional material: Pandas, Seaborn

Instructor: Márton Pósfai

Email: posfaim@ceu.edu

*Don't forget:* use the Slack channel for discussion, to ask questions, or to show solutions to exercises that are different from the ones provided in the notebook. [Slack channel](http://www.personal.ceu.edu/staff/Marton_Posfai/slack_forward.html)

In [ ]:
import pandas as pd
import numpy as np

Load the data and repeat some data processing from the main notebook.

In [ ]:
trips = pd.read_csv('Divvy_Trips_2013.csv')
trips.starttime = pd.to_datetime(trips.starttime, format="%Y-%m-%d %H:%M")
trips.stoptime = pd.to_datetime(trips.stoptime, format="%Y-%m-%d %H:%M")

In [ ]:
stations = pd.read_csv('Divvy_Stations_2013.csv')
trips2 = pd.merge(left=trips, right=stations, how='left', left_on='from_station_name', right_on='name')
trips_extended = pd.merge(trips2, stations, how='inner', left_on='to_station_name', right_on='name',
                    suffixes=['_origin', '_dest'])
del trips2

### Heterogeneous trip durations

[Video](https://ceu.cloud.panopto.eu/Panopto/Pages/Viewer.aspx?id=734d14b0-992c-41e9-bc16-ab8b00d76ffc)

For example, let's plot the trip duration distribution.

In [ ]:
trips['tripduration'].plot(kind='hist',bins=50);

This is not too helpful, we have a few trips that are much longer than average. 
Let's create a new column with the logarithm of the tripduration and plot that.

We can use the `apply()`: 

In [ ]:
trips['logduration']=trips['tripduration'].apply(np.log)

Or since `np.log()` is a numpy function, we can simply write:

In [ ]:
trips['logduration']=np.log(trips['tripduration'])

In [ ]:
ax=trips['logduration'].plot(kind='hist',bins=50,color='#9BCC31')
ax.set_xlabel('Log(trip duration)');

This looks better. What this means is that tripduration has a very wide distribution, which is closer to a lognormal than a normal.

### Most used bikes

Let's see which bike is on the road the longest: perhaps a good candidate for early retirement. The `bikeid` column uniquely identifies the bikes, to find all trips associated to each bike we can use the `groupby()` function:

In [ ]:
bike_usage = trips.groupby('bikeid')['tripduration'].sum().sort_values(ascending=False)

Let's unpack this line of code:
1. `groupby(bikeid)`: this creates a group for each individual bike
2. `['tripduration']`: pick the `tripduration` column
3. `sum()`: sum all trips for each bike, so we get a series containing the total usage and indexed by the bike IDs
4. `sort_values(ascending=False)`: sort in descending order so that the most used bike is in the first row

The most used bikes are in the first rows, we can print out the top 10 using `head()`

In [ ]:
bike_usage.head(10)

The least used bikes are at the end:

In [ ]:
bike_usage.tail(10)

We can also plot the distribution:

In [ ]:
ax = bike_usage.hist(bins=40,color='g');
ax.set_ylabel("count")
ax.set_xlabel("total usage");

### Exercise

Repeat the previous analysis, but instead of the total duration of usage, look at the total number of times the bikes were used. Do you get the same bikes in the top 10?

Previously we used the `sum()` function to aggregate the data for each bike, now you can use the `count()` function, which simply counts the number of elements in the group.

<details><summary><u>Hint</u></summary>
<p>
    
The only difference compared to the previous example is the `sum` versus `count`.

</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
bike_usage2 = trips.groupby('bikeid')['tripduration'].count().sort_values(ascending=False)
ax = bike_usage2.hist(bins=40,color='g');
ax.set_ylabel("count")
ax.set_xlabel("total number of times used");
```
    
</p>
</details>

## Optional: Plotting with Seaborn
[Video.](https://ceu.cloud.panopto.eu/Panopto/Pages/Viewer.aspx?id=2896342a-c826-4fef-8fa6-acc200ce5455)

Pandas plotting is simple but limited, matplotlib plotting is complicated but powerful. A library called [seaborn](https://seaborn.pydata.org/) is in the middle and it works very well with data in pandas. We will look at few things that we can do with it.

In [ ]:
import seaborn as sns
sns.set_style('ticks')# you can try other styles: white, dark...

(Why is seaborn imported as `sns`? [The West Wing.](https://en.wikipedia.org/wiki/Sam_Seaborn))

Do women and men use Divvy the same way? Let's take the figure that we made in the previous exercise, but separate the average trip duration for the two groups. We can use `groupby()` to group according to two columns by passing a list containing the names of the column to `groupby()` (we did something similar with the Titanic dataset)

In [ ]:
trips['dayofweek']=trips['starttime'].apply(lambda dt: dt.dayofweek) # in case you have skipped the previous exercise
dayofweek_gender_duration = trips.groupby(['dayofweek','gender'])['tripduration'].mean()

We get a series with hierarchical indices (`dayofweek` and `gender`):

In [ ]:
dayofweek_gender_duration.head()

For plotting we will need a dataframe. We can use `reset_index()` to convert the indices to regular columns and index the rows with integers instead:

In [ ]:
dayofweek_gender_duration = dayofweek_gender_duration.reset_index()
dayofweek_gender_duration.head()

As you can see, this is now a dataframe which is indexed by integers, and the previous index labels became regular columns `dayofweek` and `gender`.

We use seaborn to create a fancy plot! Let's use the `pointplot` function. If you pass the name of a column that contains a categorical value (e.g., `gender`) as the `hue` argument, the `pointplot` function will plot them as separate lines with different colors. 

In [ ]:
ax=sns.pointplot(data=dayofweek_gender_duration,x='dayofweek',y='tripduration', hue='gender')
ax.set_ylabel('Average duration of trips [s]')
ax.set_title('Average trip duration per day user gender');
# Seaborn has a bunch of formatting options
# e.g., despine removes top and right sides of the frame
#sns.despine()
#sns.despine(trim=True,offset=10)

Similar to pandas' uses matplotlib to create the plots. And similar to pandas' `plot` function, seaborn's `pointplot` also returns the current `Axes`. You can use this `Axes` to further customize your plot directly using matplotlib.

So what do we see on the figure? It seems that men go on shorter duration trips, but their trip length increases more on weekends.

### Exercise

Our dataset also has a column named `usertype`, it differentiates between customer or subscribers. Customers are one time users that pay for single trips, subscribers buy a monthly pass with unlimited rides.

Do customers and subscribers behave differently? Repeat the above analysis for these categories. In addition to trip duration, compare total number of trips as well in a separate cell. What pattern do you see?

<details><summary><u>Hint</u></summary>
<p>

You can do exatly the same as before only:
* group by `usertype` instead of `gender`
* also repeat your plot for `count()` instead of `mean()`

</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python

dayofweek_usertype_duration = trips.groupby(['dayofweek','usertype'])['tripduration'].count().reset_index()
ax=sns.pointplot(data=dayofweek_usertype_duration,x='dayofweek',y='tripduration', hue='usertype')
ax.set_ylabel('Number of trips')
ax.set_title('Number of trips per user type');
sns.despine()
```
    
</p>
</details>

We don't have to restrict ourselves to looking at the only at the daily average. Seaborn has a convinient way to plot distributions for different categories using `FacetGrid`. It creates a grid of plots where the rows and columns can represent a catergory each. Let's plot the logarithm of the tripduration for each day and the two usertypes.

(To show the histograms, we are going to use the [kernel density estimate](https://en.wikipedia.org/wiki/Kernel_density_estimation) plot of seaborn, which is basically a smoothed version of the raw histogram.)

In [ ]:
g = sns.FacetGrid(trips, col='dayofweek', row='usertype') #specify the grid
g.map(sns.kdeplot, 'logduration') #specify the type of plot, here: kernel density estimate

We can also use the color of the lines to represent a category:

In [ ]:
g = sns.FacetGrid(trips, col='dayofweek', hue='usertype', col_wrap=4) #col_wrap
g.map(sns.kdeplot, 'logduration') #specify the type of plot, here: kernel density estimate
g.add_legend(); #add legend

Therefore, using rows, columns, and hue we can represent three types of categories simultaneously.

[Video.](https://ceu.cloud.panopto.eu/Panopto/Pages/Viewer.aspx?id=bb03d79f-e724-4060-9b54-acc200d33de6)

This figure looks okay, but we can make this better. Uncomment the lines one-by-one in the next cell to see how we format the figure.

In [ ]:
#sns.set_style("white", {"axes.facecolor": (1.,1.,1.,0.)}) # changes the face to transparent globally
#not only this figure
pal = sns.cubehelix_palette(7, start=2.8, rot=.75, light=.7) # define a new color palette
g = sns.FacetGrid(trips, row='dayofweek',col='usertype',
                  #hue='dayofweek', # color the rows differently
                  #aspect=6, #aspect ratio of subplots
                  #height=1., #height of subplots 
                  #margin_titles=True, #indicate categories at the top and end of columns and rows
                  #palette=pal #use new palette
                 )
g.map(sns.kdeplot, 'logduration',
      #shade=True, #color area under the curve
      #alpha=.5,   # make color semitransparent
      linewidth=1.5 
     ) 

#g.set_titles(row_template = '                        ', col_template = '{col_name}') #change how categories are printed
# the spaces are needed because of a bug in the windows version of seaborn 
#g.add_legend(); #add legend to indicate the days instead of row labels

#g.set(xlim=(4,10)) # set the range of x displayed to focus on the bulk of the distribution
#g.set(xticks=[np.log(100),np.log(1000),np.log(10000)]) #change location of ticks
#g.set(xticklabels=['log(100)','log(1000)','log(10000)']) #and labels to be more informative


#g.fig.subplots_adjust(hspace=-.5) #make subplots overlap

# remove axes details that don't play well with overlap
#g.set(yticks=[]) # remove y axis and its trappings
#the y axis is shared so we can still compare the shape of the distributions
#g.despine(bottom=True, left=True)



I am not sure if this looks good or just unusual, but at least we demonstrated some formatting options. Check out the [gallery of examples](https://seaborn.pydata.org/examples/index.html) and the [tutorials](https://seaborn.pydata.org/tutorial.html) to get inspired.

To color the plots above we used a [cubehelix palette](https://seaborn.pydata.org/generated/seaborn.cubehelix_palette.html). This is great palette to use for heat maps too, because the intensity of the color monotonically changes through the palette, meaning that it prints well on black-and-white printers, and even people with any type of colorblindness are able to read it. For more details check out this [blog post](https://ifweassume.blogspot.com/2013/05/cubehelix-or-how-i-learned-to-love.html).

### Distance between stations

Our dataset does not contain information about the trip length, but we do have the latitude and longitude coordinates of the stations. This allows us to calculate the distance between the origin and destination stations, but note that this is not the distance you have to travel on the road, but distance as the crow flies.

To do this we have to recall an old favorite: the [haversine formula](https://en.wikipedia.org/wiki/Haversine_formula) which takes a pair of latitude-longitude coordinates and returns their distance in kilometeres:

In [ ]:
import math
def latlongdist(lat1,long1,lat2,long2):
    rlat1 = math.radians(lat1)
    rlat2 = math.radians(lat2)
    rlong1 = math.radians(long1)
    rlong2 = math.radians(long2)
    dlat = rlat2 - rlat1
    dlong = rlong2 - rlong1
    a = math.sin(dlat / 2)**2 + math.cos(rlat1) * math.cos(rlat2) * math.sin(dlong / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return 6371.0 * c

print(latlongdist(48.105625, 20.790556, 46.07308, 18.22857))

Let's create a new column for the `trips_extended` dataframe that contains the distance using the `apply()` method of the dataframe. Remember in this case the function that we apply has to take the entire row as an input (see previous class, if you don't remember). However, `latlongdist(lat1,long1,lat2,long2)` takes four floats as an input and not the row. How can we solve this? With a lambda function of course!

In [ ]:
trips_extended['dist']=trips_extended.apply(lambda row: 
                                            latlongdist(row['latitude_origin'],
                                                        row['longitude_origin'],
                                                        row['latitude_dest'],
                                                        row['longitude_dest']),
                                           axis=1)

The above lambda function takes a row of the dataframe as input, selects the coordinates and calles the `latlongdist` function.

Reminder: `axis=1` indicates that we apply the function to each row, `axis=0` would mean applying the function to each column.

Now we let's investigate the new `dist` column!

### Exercise

Make a histogram of the distances between origin and destination stations using pandas' plotting capability. Try different number of bins. Try to set the y axis to logarithmic scale using `ax.set_yscale('log')`. What kind of patterns do you see?

<details><summary><u>Hint.</u></summary>
<p>

To set the y axis to logarithmic scale you have to grab the current `Axes` object. Luckily it is returned by pandas' plot function, so you can write:
```python
ax=trips_extended['dist'].plot(...)
```

</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
ax=trips_extended['dist'].plot(kind='hist',bins=200)
#ax.set_yscale('log')
```
Looking at the histogram with enough bins it becomes apparent that there is large a peak at zero. This corresponds to trips where the user returned the bike to the same station where they got it from; therefore the origin and destination station is the same.
    
If we chang the y axis to logarithmic scale, the tail of the distribution becomes a straight line, indicating that the tail decays as an exponential function. The distance distribution is not as heterogeneous as the trip duration distribution.

</p>
</details>

It is reasonable to assume that there is a correlation between station distance and trip duration: the larger the distance the more time is needed for the trip. We could use pandas' for plotting but let's do something more fancy and use seaborn.

What is a good plot to visualize the hypothesised correlation? A scatter plot is probably not a good idea, since there are >700,000 trips which would create too many markers. A two dimensional histogram is a better option.

We use the `jointplot` function of seaborn which allows to show both the relation between two columns and their marginal distribution:

In [ ]:
g = sns.jointplot(
    data=trips_extended,
    x="logduration", y="dist",
    kind="hex",
    xlim=(4,9),
    ylim=(0,8)
)

Seaborn has several plotting options to visualize the relation of two variables, we chose `hex` which is a two dimensional histogram with hexagonal bins. We also set the range of the plot on the x and y axes using `xlim` and `ylim` to focus on the bulk of the distributions (try commenting out those lines). See further plotting options [here](https://seaborn.pydata.org/generated/seaborn.jointplot.html#seaborn.jointplot).

So what do we see on the plot? There is a positive correlation as we expected and it seems to be nonlinear. (Remember, we are comparing the logarithm of the trip duration to as-the-crow-flies distance, there is no real reason to expect the correlation to be linear.) An apparent outlier is the zero distance, there are very long trips where the bikes are returned to the origin station.

Let's further investigate! If we use `kind=kde` (aka kernel density estimate) instead of `hex`, we can add an additional dimension to the plot: the color of the contours is used to indicate categories. For example, we can compare the occassional customers to subscribers by setting `hue="usertype".

(Before you run the next cell: calculating the kde for 700,000 datapoints takes a few minutes. If you don't have that time or you would like to experiment with the plotting options, select a subset of rows for quicker results.)

In [ ]:
g = sns.jointplot(
    data=trips_extended#[:10000],#apply this slice for quicker plotting
    x="logduration", y="dist", hue="usertype",
    kind="kde",
    xlim=(4,9),
    ylim=(0,8),
    marginal_kws={'fill':True,'alpha':.5}
)

We used the `marginal_kws` to pass on additional formatting settings as a dictionary to the marginal plots: we colored the area under the curves and set them to be transparent.

### Exercise

Does the `logduration` distribution depend whether it is a weekend or a weekday? Does it depend on gender? Does it depend if the bike was returned to the same station? Answer these questions doing the following steps:
* Create a column indicating if the trip was on a weekend.
* Create a column indicating that the bike was returned to the same station.
* Create a facet grid using seaborn where rows indicate gender, columns indicate whether the trips starts and ends at the same station, and hue indicates weekend or weekday.

<details><summary><u>Hint.</u></summary>
<p>

You can create the new columns with
```python
trips_extended['weekend']=trips_extended['dayofweek']>4 #5: Saturday, 6: Sunday
trips_extended['samestation']=trips_extended['from_station_id']==trips_extended['to_station_id']
```

</p>
</details>

<details><summary><u>Solution.</u></summary>
<p>
    
```python
trips_extended['weekend']=trips_extended['dayofweek']>4 #5: Saturday, 6: Sunday
trips_extended['samestation']=trips_extended['from_station_id']==trips_extended['to_station_id']

g = sns.FacetGrid(trips_extended, row='gender',hue='weekend',col='samestation',
                  sharex=True,
                  sharey=True,
                  aspect=2, 
                  height=2., 
                  margin_titles=True,
                 ) #specify the grid
g.map(sns.kdeplot, 'logduration', linewidth=1.5) #specify the type of plot, here: kernel density estimate

g.set(xlim=(3,10))
g.set(xticks=[np.log(10),np.log(100),np.log(1000),np.log(10000)])
g.set(xticklabels=['log(10)','log(100)','log(1000)','log(10000)'])
g.add_legend(); #add legend
```

</p>
</details>